In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import ollama

# List installed models
print(ollama.list())

# Test text generation
r = ollama.chat(
    model='llama3.2:3b',  # small text model, fast
    messages=[{'role': 'user', 'content': 'Say hello in exactly 3 words.'}]
)
print(r['message']['content'])

In [ ]:
import ollama

png_path = "/home/graham/Tracking-Analysis/data/control/ads/control500/7517315819367483/91aceee53f36.png"

response = ollama.chat(
    model='hf.co/vinimuchulski/gemma-3-12b-it-qat-q4_0-gguf:latest',
    messages=[{
        'role': 'user',
        'content': 'Describe this advertisement in 1-2 sentences. What product or service is being advertised, and what is the visual style?',
        'images': [str(png_path)]
    }]
)
description = response['message']['content']

In [ ]:
print(str(response))

In [ ]:
import ollama

# Small text-only test first — should be nearly instant
r = ollama.chat(
    model='llama3.2:3b' if False else 'llama3.1:latest',  # use what you have
    messages=[{'role': 'user', 'content': 'Reply with just: OK'}]
)
print("Text model:", r['message']['content'])

# Vision test — this is the moment of truth for the mllama fix
r = ollama.chat(
    model='llama3.2-vision:11b',
    messages=[{
        'role': 'user',
        'content': 'What do you see? Reply in one sentence.',
        'images': ['data/control/ads/control500/138214581715761/24e31d056c11.png']
        # ↑ swap in any real ad path from your dataset
    }]
)
print("Vision model:", r['message']['content'])

In [ ]:
categories = [
    "Automotive",
    "Beauty & Personal Care",
    "Business Services",
    "Construction & Home Improvement",
    "Consumer Electronics",
    "Education",
    "Energy & Utilities",
    "Entertainment",
    "Fashion & Apparel",
    "Finance",
    "Food & Beverage",
    "Gaming",
    "Government & Public Services",
    "Health & Wellness",
    "Healthcare",
    "Home & Garden",
    "Industrial & Manufacturing",
    "Insurance",
    "Jewelry & Luxury Goods",
    "Legal Services",
    "Marketplace & Classifieds",
    "Media & Publishing",
    "Nonprofit & Charity",
    "Pets",
    "Real Estate",
    "Recruitment & Careers",
    "Restaurants & Dining",
    "Retail",
    "Software & SaaS",
    "Sports & Fitness",
    "Technology",
    "Telecommunications",
    "Travel & Hospitality",
    "Transportation & Logistics",
    "Consumer Packaged Goods",
    "Cryptocurrency & Web3",
    "Dating",
    "Events & Conferences",
    "Parenting & Family",
    "Photography & Creative Services",
    "Religion & Faith",
    "Security & Privacy",
    "Smart Home & IoT",
    "Streaming Services",
    "Subscription Services",
    "Toys & Hobbies",
    "Adult",
    "Political",
    "Public Safety",
    "Other"
]
print(f"""You are analyzing an image that may or may not contain an 
advertisement, captured during a web privacy study.

FIRST, evaluate the image:
- If the image is mostly blank, contains only a fragment of an ad, 
  is unreadable, is only a play button, or does not show a coherent advertisement, respond with 
  EXACTLY this JSON and nothing else:
  {{"is_valid_ad": false, "reason": "<brief explanation>"}}
- If the image only contains an image of a large button with phrasing like 'Continue' or 'Download',
  the ad is a scam. Respond with the following JSON with this in mind.

- If the image DOES contain a coherent, readable advertisement, respond with:
{{
  "is_valid_ad": true,
  "category": "<Specify a category from list: {categories}>",
  "primary_product_or_service": "<what is being sold>",
  "advertiser_brand": "<brand name if visible, otherwise 'unknown'>",
  "visual_description": "<one sentence describing the imagery>",
  "text_content": "<any headline/CTA text visible>",
  "confidence": "<'high', 'medium', or 'low' — how sure are you>"
}}


Return ONLY the JSON. Do not include any other text or reasoning.
""")

In [ ]:
import src.analysis.ad_clip_ollama as o

ad_hashes = [
    "4a9670eadccd", # Blank white image with "advertisement" text at top
    "e4d2368c7673", # Small car video ad over article text
    "a99d115ee0b3", # Different language
    "2d0c41fbe9fc", # Dark overlay of article, not an ad
    "678d9e4b78e0", # Scam
    "82b2056b7028", # Cut off banner ad
    "462ef2a17aa9", # Unclear sponsors
    "2d1175d49918", # Download button scam
]

o.run_clip_pipeline(single_ad=list(ad_hashes))

In [ ]:
import pandas as pd
ads = pd.read_parquet(o.ADS_PARQUET)
ads = ads.loc[ads['ad_hash'].isin(ad_hashes)]
print(ads.count)